# Notebook 05 Batch Inference, Evaluasi Otomatis, dan Plot

Notebook ini menjalankan batch inference untuk ground truth, menghitung metrik retrieval dan kualitas jawaban, menyimpan tabel evaluasi, membuat review kegagalan, dan menghasilkan plot otomatis.

In [ ]:
# Jalankan bila package belum ada
# !pip install -q chromadb rank_bm25 sentence-transformers transformers accelerate bitsandbytes pandas numpy matplotlib scikit-learn openpyxl tabulate tqdm

In [ ]:
import os, re, json, time, pickle, math
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import torch
import chromadb
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")
GROUND_TRUTH_CSV = Path("../data/ground_truth_eval_20.csv")
OUTPUT_DIR = Path("../data/eval_outputs")
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
SEMANTIC_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
USE_RERANKER = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device", DEVICE)
print("ground truth", GROUND_TRUTH_CSV.resolve())
print("output", OUTPUT_DIR.resolve())

In [ ]:
def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]+|[\u00C0-\u024F\u1E00-\u1EFF]+|[\w]+", str(text).lower())


def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


def load_chunks(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Chunk tidak ditemukan: {path}. Jalankan notebook 01 untuk embedding dan indexing dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    for i, c in enumerate(chunks[:10]):
        for k in ["id", "text", "display_text", "embedding_text", "citation_text", "metadata"]:
            if k not in c:
                raise ValueError(f"Schema chunk belum sesuai. Field hilang: {k} pada index {i}")
    return chunks

chunks = load_chunks(DATA_PATH)
id_to_chunk = {c["id"]: c for c in chunks}
print("chunks", len(chunks))

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)
print("chroma count", collection.count())
if collection.count() != len(chunks):
    raise ValueError(f"Chroma count {collection.count()} tidak sama dengan chunks {len(chunks)}. Jalankan notebook 01 dulu.")

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload.get("bm25")
    bm25_ids = payload.get("ids", [])
    print("bm25", len(bm25_ids))
else:
    print("BM25 tidak ditemukan. Dense retrieval tetap berjalan.")

In [ ]:
MAX_CONTEXT_DOCS = 4
MIN_QUERY_TERM_OVERLAP = 1

LEGAL_QUERY_EXPANSIONS = [
    {
        "triggers": ["pelanggaran berat", "mendesak", "bersifat mendesak"],
        "expansion": (
            "pelanggaran bersifat mendesak uang pisah uang penggantian hak "
            "tidak mendapat pesangon tidak mendapat uang penghargaan masa kerja"
        ),
    },
    {
        "triggers": ["surat peringatan", "sp pertama", "sp kedua", "sp ketiga"],
        "expansion": (
            "pelanggaran ketentuan surat peringatan pertama kedua ketiga "
            "pesangon nol koma lima uang penghargaan masa kerja uang penggantian hak"
        ),
    },
    {
        "triggers": ["pesangon", "phk", "pemutusan hubungan kerja"],
        "expansion": (
            "pemutusan hubungan kerja uang pesangon uang penghargaan masa kerja "
            "uang penggantian hak hak akibat pemutusan hubungan kerja"
        ),
    },
    {
        "triggers": ["pkwt", "kontrak"],
        "expansion": (
            "perjanjian kerja waktu tertentu kompensasi pkwt "
            "jangka waktu perpanjangan pembaruan"
        ),
    },
    {
        "triggers": ["alih daya", "outsourcing", "outsourced"],
        "expansion": (
            "alih daya perusahaan alih daya pekerja buruh "
            "hubungan kerja perlindungan upah kesejahteraan"
        ),
    },
    {
        "triggers": ["upah", "gaji", "tunjangan"],
        "expansion": (
            "upah gaji tunjangan tetap struktur skala upah "
            "upah minimum pembayaran upah"
        ),
    },
]

KETENAGAKERJAAN_POSITIVE_KEYWORDS = {
    "ketenagakerjaan", "tenaga kerja", "pekerja", "buruh", "hubungan kerja",
    "perjanjian kerja", "pemutusan hubungan kerja", "phk", "pesangon",
    "upah", "pengupahan", "pensiun", "jaminan sosial", "jaminan kerja",
    "alih daya", "outsourcing", "pkwt", "pkwtt", "serikat pekerja",
    "pengusaha", "perusahaan alih daya", "perlindungan pekerja",
    "waktu kerja", "cuti", "k3", "keselamatan kerja", "cipta kerja",
}

KETENAGAKERJAAN_NEGATIVE_KEYWORDS = {
    "perizinan berusaha", "oss", "risiko usaha", "izin usaha", "nib",
    "investasi", "badan usaha", "penyelenggaraan usaha", "sistem perizinan",
    "rba", "risk based approach", "sektor usaha", "kbli",
    "penyelenggaraan pemerintahan", "administrasi pemerintahan",
    "sop administrasi", "pelayanan publik",
}

KETENAGAKERJAAN_KNOWN_REGS = {
    ("pp", "35"), ("pp", "36"), ("pp", "34"), ("pp", "45"),
    ("uu", "13"), ("uu", "6"), ("uu", "24"), ("uu", "1"), ("uu", "21"),
    ("permen", "5"), ("permen", "6"),
}


def normalize_reg_type(value: str) -> str:
    value = str(value or "").lower()
    if "undang" in value or value == "uu":
        return "uu"
    if "pemerintah" in value or value == "pp":
        return "pp"
    if "presiden" in value or "perpres" in value:
        return "perpres"
    if "menteri" in value or "permen" in value:
        return "permen"
    return value


def is_ketenagakerjaan_doc(meta: Dict[str, Any]) -> bool:
    tentang = str(meta.get("tentang", "") or "").lower()
    bab_title = str(meta.get("bab_title", "") or "").lower()
    bagian_title = str(meta.get("bagian_title", "") or "").lower()
    haystack = f"{tentang} {bab_title} {bagian_title}"
    if any(neg in haystack for neg in KETENAGAKERJAAN_NEGATIVE_KEYWORDS):
        return False
    reg_type = normalize_reg_type(meta.get("regulation_type", ""))
    nomor = str(meta.get("nomor", "") or "").strip()
    if (reg_type, nomor) in KETENAGAKERJAAN_KNOWN_REGS:
        return True
    if any(pos in haystack for pos in KETENAGAKERJAAN_POSITIVE_KEYWORDS):
        return True
    return not tentang.strip()


def expand_query_terms(query: str) -> str:
    q = query.lower()
    expansions = []
    for rule in LEGAL_QUERY_EXPANSIONS:
        if any(trigger in q for trigger in rule["triggers"]):
            expansions.append(rule["expansion"])
    return normalize_text(" ".join([query] + expansions))


def retrieval_queries(query: str) -> List[str]:
    expanded = expand_query_terms(query)
    return [query, expanded] if expanded != query else [query]


def dense_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(["query: " + query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=fetch_k, include=["documents", "metadatas", "distances"])
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    if bm25 is None or not bm25_ids:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights=None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
            item["hit"].update({k: v for k, v in hit.items() if k not in item["hit"]})
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = float(item["score"])
        out.append(hit)
    return out


def lex_posterior_score(hit: Dict[str, Any]) -> float:
    meta = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))
    try:
        year = int(meta.get("publication_year") or meta.get("year") or 0)
    except Exception:
        year = 0
    try:
        hierarchy = int(meta.get("regulation_hierarchy") or 99)
    except Exception:
        hierarchy = 99
    if str(meta.get("active_status", "")).lower() == "berlaku":
        score += 0.030
    if is_ketenagakerjaan_doc(meta):
        score += min(max(year - 2000, 0), 40) * 0.001
    else:
        score -= 0.100
    score += max(0, 6 - hierarchy) * 0.003
    if meta.get("quality_status") == "needs_review":
        score -= 0.010
    return score


def dedupe_legal_hits(hits: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    ranked = sorted(hits, key=lex_posterior_score, reverse=True)
    seen, out = set(), []
    for hit in ranked:
        meta = hit.get("metadata", {})
        if not is_ketenagakerjaan_doc(meta):
            continue
        key = (meta.get("source_file", ""), meta.get("pasal_id", ""), meta.get("chunk_kind", ""), meta.get("chunk_index", ""))
        if key in seen:
            continue
        seen.add(key)
        hit["final_score"] = lex_posterior_score(hit)
        out.append(hit)
        if len(out) >= k:
            break
    return out


reranker = None
if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
    print("Reranker aktif:", RERANKER_MODEL_NAME)
else:
    print("Reranker neural nonaktif. Retrieval memakai RRF dense+BM25 + dedupe legal.")


def rerank_documents(query: str, docs: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    if reranker is None or not docs:
        return docs[:k]
    scores = reranker.predict([(query, d["text"]) for d in docs])
    for d, s in zip(docs, scores):
        d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x.get("rerank_score", 0.0), reverse=True)[:k]


def retrieve_documents(query: str, k: int = 8, fetch_k: int = 40, use_bm25: bool = True) -> List[Dict[str, Any]]:
    dense_hits = dense_search(query, fetch_k=fetch_k)
    sparse_hits = bm25_search(query, fetch_k=fetch_k) if use_bm25 else []
    fused = rrf_fuse([dense_hits, sparse_hits], weights=[1.0, 0.7]) if sparse_hits else dense_hits
    return dedupe_legal_hits(fused, k=k)


def query_terms(query: str) -> set[str]:
    stopwords = {
        "yang", "dan", "atau", "karena", "dengan", "untuk", "pada", "dalam",
        "jika", "maka", "dari", "berapa", "apakah", "bagaimana", "dimana",
        "kapan", "siapa", "pekerja", "buruh", "pengusaha", "perusahaan", "hak",
        "nya", "itu", "ini", "ada", "dapat", "bisa", "oleh", "ke", "di", "atas",
    }
    return {t for t in re.findall(r"[a-zA-Z0-9]+", query.lower()) if len(t) > 2 and t not in stopwords}


def doc_relevance_score(query: str, doc: Dict[str, Any]) -> float:
    terms = query_terms(expand_query_terms(query))
    if not terms:
        return 1.0
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")), str(meta.get("citation_text", "")), str(meta.get("regulation_type", "")),
        str(meta.get("nomor", "")), str(meta.get("pasal_id", "")), str(meta.get("bab_title", "")), str(meta.get("bagian_title", "")),
    ]).lower()
    score = sum(1 for term in terms if term in haystack) / max(len(terms), 1)
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    if specific_terms and not any(term in haystack for term in specific_terms):
        score -= 0.25
    if "final_score" in doc:
        score += min(float(doc.get("final_score", 0.0)), 1.0) * 0.05
    elif "rrf_score" in doc:
        score += min(float(doc.get("rrf_score", 0.0)), 1.0) * 0.05
    return score


def has_original_specific_overlap(query: str, doc: Dict[str, Any]) -> bool:
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    if not specific_terms:
        return True
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")), str(meta.get("citation_text", "")), str(meta.get("bab_title", "")), str(meta.get("bagian_title", "")),
    ]).lower()
    return any(term in haystack for term in specific_terms)


def filter_relevant_context(query: str, docs: List[Dict[str, Any]], max_docs: int = MAX_CONTEXT_DOCS) -> List[Dict[str, Any]]:
    terms = query_terms(expand_query_terms(query))
    scored = []
    for doc in docs:
        if not has_original_specific_overlap(query, doc):
            continue
        score = doc_relevance_score(query, doc)
        overlap = int(round(max(score, 0) * max(len(terms), 1))) if terms else 1
        if (not terms or overlap >= MIN_QUERY_TERM_OVERLAP) and score > 0:
            doc["context_relevance_score"] = score
            scored.append(doc)
    if not scored:
        return docs[:1]
    return sorted(scored, key=lambda d: d.get("context_relevance_score", 0.0), reverse=True)[:max_docs]


def retrieve_context(query: str, k: int = 8, fetch_k: int = 50) -> List[Dict[str, Any]]:
    candidates = []
    seen = set()
    for q in retrieval_queries(query):
        hits = retrieve_documents(q, k=max(k * 6, 18), fetch_k=fetch_k)
        for hit in hits:
            key = hit.get("id") or (
                hit.get("metadata", {}).get("source_file", ""),
                hit.get("metadata", {}).get("pasal_id", ""),
                hit.get("metadata", {}).get("chunk_index", ""),
            )
            if key in seen:
                continue
            seen.add(key)
            candidates.append(hit)
    reranked = rerank_documents(expand_query_terms(query), candidates, k=max(k * 6, 18))
    return filter_relevant_context(query, reranked, max_docs=k)


def build_reference(meta: Dict[str, Any]) -> str:
    citation = meta.get("citation_text") or meta.get("citation") or ""
    source = meta.get("source_file") or meta.get("file_name") or ""
    pasal = meta.get("pasal_id") or meta.get("article") or ""
    return " | ".join([str(p) for p in [citation, source, pasal] if str(p).strip()])


def build_context(docs: List[Dict[str, Any]], max_docs: int | None = None) -> str:
    selected = docs[:max_docs] if max_docs else docs
    blocks = []
    for i, d in enumerate(selected, 1):
        meta = d.get("metadata", {})
        ref = build_reference(meta) or f"Dokumen {i}"
        text = normalize_text(d.get("text", ""))[:2500]
        blocks.append(f"SUMBER HUKUM {i}: {ref}\n{text}")
    return "\n\n".join(blocks)


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = os.getenv("RAG_LLM_MODEL_ID", "Qwen/Qwen3.5-9B")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "700"))
USE_4BIT = os.getenv("USE_4BIT", "1") == "1" and torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("model ready", MODEL_ID)

In [ ]:
def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*(analysis|reasoning)\s*:\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def sanitize_output(text: str) -> str:
    text = strip_thinking(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def apply_chat_template_no_thinking(messages: List[Dict[str, str]]) -> str:
    if hasattr(processor, "apply_chat_template"):
        try:
            return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages]) + "\nassistant:"


def generate_chat_text(messages: List[Dict[str, str]], max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = apply_chat_template_no_thinking(messages)
    inputs = processor(text=prompt, return_tensors="pt", padding=True, truncation=True, max_length=12000).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.04,
            top_p=0.9,
            eos_token_id=getattr(processor, "eos_token_id", None),
            pad_token_id=getattr(processor, "eos_token_id", None),
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = processor.decode(new_ids, skip_special_tokens=True)
    return strip_thinking(text)


CONDITIONAL_LOGIC_GUARDRAIL = """
ATURAN KRITIS - PEMISAHAN KONDISI HUKUM (WAJIB DIPATUHI):
Saat dokumen memuat BEBERAPA AYAT dengan kondisi berbeda dalam satu pasal yang sama,
kamu WAJIB mengidentifikasi ayat mana yang berlaku untuk kasus pengguna, lalu HANYA
gunakan konsekuensi hukum dari ayat tersebut. DILARANG KERAS menggabungkan kondisi
dari dua ayat berbeda menjadi satu narasi.

Contoh penerapan pada Pasal 52 PP No. 35 Tahun 2021:
  KONDISI A - Ayat (1): PHK karena pelanggaran ketentuan yang didahului SP ke-1, ke-2,
    dan ke-3. Konsekuensi: pesangon 0,5x, UPMK 1x, UPH. JANGAN sebut ini
    "pelanggaran berat". JANGAN sebut konsekuensi ini untuk kondisi mendesak.
  KONDISI B - Ayat (2)/(4): PHK karena pelanggaran bersifat mendesak (dahulu disebut
    pelanggaran berat). Dilakukan TANPA SP 1,2,3. Konsekuensi: TIDAK BERHAK PESANGON,
    TIDAK BERHAK UPMK, hanya berhak UPH dan Uang Pisah.
  LARANGAN ABSOLUT: Jangan pernah menulis "pelanggaran berat" lalu menyebut
    "pesangon 0,5x" karena itu kontradiksi fatal. Pilih satu kondisi sesuai pertanyaan.

Prinsip ini berlaku umum untuk SEMUA pasal yang memiliki beberapa ayat dengan
kondisi berbeda di seluruh dokumen hukum yang tersedia.
"""

SYSTEM_PROMPT = (
    "Kamu adalah pakar hukum ketenagakerjaan Indonesia yang sangat teliti.\n"
    "Saat membaca dokumen hukum, kamu harus memahami bahwa setiap AYAT memiliki kondisi (syarat) dan konsekuensi yang BERBEDA.\n\n"
    "ATURAN UTAMA:\n"
    "1. JAWAB HANYA berdasarkan KONTEKS. DILARANG mengarang atau memakai pengetahuan luar.\n"
    "2. Langsung sebutkan sumber hukumnya secara natural, contoh: Peraturan Pemerintah No. 35 Tahun 2021, Pasal 52 ayat (2).\n"
    "   Sertakan nomor AYAT jika relevan karena ini penting untuk membedakan kondisi hukum yang berbeda.\n"
    "3. DILARANG memakai kode [R1], [R2], SUMBER HUKUM 1, atau ID internal lain di jawaban.\n"
    "4. DILARANG menulis daftar referensi di dalam jawaban; sistem akan mencetak referensi terpisah.\n"
    "5. PENANGANAN KETERBATASAN INFORMASI:\n"
    "   a. Jika pertanyaan menggunakan istilah lama, bahasa awam, atau sinonim "
    "(contoh: 'pelanggaran berat' = 'pelanggaran bersifat mendesak', "
    "'kontrak' = 'PKWT', 'dipecat' = 'PHK'), JANGAN tolak pertanyaan. "
    "Petakan ke istilah resmi dalam dokumen, sebutkan perubahan istilah tersebut secara singkat di awal jawaban, lalu langsung jawab substansinya.\n"
    "   b. Hanya gunakan kalimat 'Maaf, informasi tersebut tidak tersedia dalam database hukum ketenagakerjaan yang saya miliki.' "
    "jika setelah memetakan semua kemungkinan sinonim pun tidak ada dokumen yang relevan sama sekali.\n"
    "   c. DILARANG KERAS memulai jawaban dengan kata 'Maaf' atau kalimat disclaimer apapun jika konteks sudah tersedia. Langsung jawab substansinya.\n"
    "6. Selalu utamakan aturan terbaru atau aturan yang lebih spesifik jika ada perbedaan antar referensi.\n"
    "7. KETAT PADA KONTEKS: Abaikan dokumen atau pasal yang tidak relevan dengan substansi pertanyaan pengguna.\n"
    "8. SPESIFIK & AKURAT: Sebutkan peraturan, Pasal, beserta AYAT-nya dengan presisi. Jangan pernah mencampuradukkan konsekuensi antar ayat!\n"
    "9. KOMPREHENSIF: Sebutkan semua komponen hak, kewajiban, atau tata cara secara lengkap sesuai ayat yang diekstrak.\n"
    "   Jika berdasarkan pasal tersebut ada ketentuan 'tidak mendapat X', nyatakan pengecualian tersebut dengan tegas.\n"
    "10. Bahasa Indonesia harus rapi, baku, tanpa typo, tanpa campuran aksara asing, tanpa markdown, dan tidak berulang.\n"
    "11. Jika ragu, lebih baik jawab keterbatasan konteks daripada membuat pasal/tahun palsu.\n"
    f"{CONDITIONAL_LOGIC_GUARDRAIL}"
)


def build_messages(question: str, docs: List[Dict[str, Any]], max_context_docs: int = 6) -> List[Dict[str, str]]:
    context = build_context(docs, max_docs=max_context_docs)
    user_prompt = f"KONTEKS REFERENSI HUKUM:\n{context}\n\nPERTANYAAN PENGGUNA:\n{question}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def generate_answer(question: str, k: int = 8, max_context_docs: int = 6) -> Dict[str, Any]:
    t0 = time.time()
    docs = retrieve_context(question, k=k, fetch_k=50)
    answer = sanitize_output(generate_chat_text(build_messages(question, docs, max_context_docs=max_context_docs)))
    latency = time.time() - t0
    refs = []
    for i, d in enumerate(docs, 1):
        meta = d.get("metadata", {})
        refs.append({
            "rank": i,
            "chunk_id": d.get("id", ""),
            "reference": build_reference(meta),
            "source_file": meta.get("source_file", ""),
            "pasal_id": meta.get("pasal_id", ""),
            "rrf_score": d.get("rrf_score", None),
            "rerank_score": d.get("rerank_score", None),
            "context_relevance_score": d.get("context_relevance_score", None),
            "text_preview": normalize_text(d.get("text", ""))[:500],
        })
    return {"question": question, "answer": answer, "references": refs, "latency_seconds": latency}


In [ ]:
gt = pd.read_csv(GROUND_TRUTH_CSV)
required_cols = {"id", "topic", "question", "expected_answer", "expected_keywords", "expected_articles", "expected_law_numbers", "expected_citations"}
missing = required_cols - set(gt.columns)
if missing:
    raise ValueError(f"Kolom ground truth hilang: {sorted(missing)}")

results = []
for row in tqdm(gt.to_dict("records"), total=len(gt), desc="batch inference"):
    out = generate_answer(row["question"], k=8, max_context_docs=6)
    refs = out["references"]
    results.append({
        **row,
        "model_id": MODEL_ID,
        "answer": out["answer"],
        "latency_seconds": out["latency_seconds"],
        "retrieved_references_json": json.dumps(refs, ensure_ascii=False),
        "retrieved_references_text": " || ".join([r["reference"] for r in refs]),
        "top1_reference": refs[0]["reference"] if refs else "",
        "retrieved_count": len(refs),
    })

df = pd.DataFrame(results)
res_csv = OUTPUT_DIR / "rag_inference_results.csv"
res_jsonl = OUTPUT_DIR / "rag_inference_results.jsonl"
df.to_csv(res_csv, index=False, encoding="utf-8-sig")
with res_jsonl.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("saved", res_csv.resolve())
print("saved", res_jsonl.resolve())
df[["id", "topic", "question", "answer", "latency_seconds", "top1_reference"]].head()

In [ ]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).lower()).strip()


def compact(s: str) -> str:
    return re.sub(r"[^0-9a-z]+", "", norm(s))


def split_targets(s: str) -> List[str]:
    # Multi-target fields use semicolon. Commas are part of chunk citation_text, e.g. "..., Pasal 52".
    return [x.strip() for x in str(s).split(";") if x.strip() and x.strip().lower() != "nan"]


def keyword_hits(answer: str, keywords: str) -> Dict[str, Any]:
    ks = split_targets(keywords)
    a = norm(answer)
    ac = compact(answer)
    hits = []
    for k in ks:
        nk = norm(k)
        kc = compact(k)
        digit = re.sub(r"[^0-9]", "", nk)
        ok = nk in a or (kc and kc in ac) or (digit and digit in re.sub(r"[^0-9]", "", a))
        if ok:
            hits.append(k)
    return {"hits": hits, "missing": [k for k in ks if k not in hits], "score": len(hits) / len(ks) if ks else 0.0}


def target_variants(target: str) -> List[str]:
    t = norm(target)
    variants = {t}
    m = re.search(r"\b(pp|uu)\s*(\d+)\s*/\s*(\d{4})\b", t)
    if m:
        kind, number, year = m.groups()
        long_kind = "peraturan pemerintah" if kind == "pp" else "undang undang"
        variants.add(f"{long_kind} no {number} tahun {year}")
        variants.add(f"{long_kind} nomor {number} tahun {year}")
    m = re.search(r"\bpermenaker\s*(?:nomor\s*)?(\d+)\s*/\s*(\d{4})\b", t)
    if m:
        number, year = m.groups()
        variants.add(f"peraturan menteri ketenagakerjaan no {number} tahun {year}")
        variants.add(f"peraturan menteri ketenagakerjaan nomor {number} tahun {year}")
    m = re.search(r"\bpermenaker\s+per\.?(\d+)\s*/\s*men\s*/\s*([ivxlcdm]+)\s*/\s*(\d{4})\b", t)
    if m:
        number, roman, year = m.groups()
        variants.add(f"peraturan menteri ketenagakerjaan no per {number} men {roman} {year}")
        variants.add(f"peraturan menteri ketenagakerjaan no per {number} men {roman.upper()} {year}".lower())
    return [v for v in variants if v]


def contains_target(blob: str, target: str) -> bool:
    b = norm(blob)
    bc = compact(blob)
    bd = re.sub(r"[^0-9]", "", b)
    for variant in target_variants(target):
        vc = compact(variant)
        vd = re.sub(r"[^0-9]", "", variant)
        if variant in b or (vc and vc in bc) or (vd and len(vd) >= 3 and vd in bd):
            return True
    return False


def contains_any(blob: str, values: List[str]) -> int:
    return int(any(contains_target(blob, v) for v in values))


def target_coverage(blob: str, values: List[str]) -> float:
    targets = [v for v in values if str(v).strip()]
    if not targets:
        return 0.0
    return sum(1 for v in targets if contains_target(blob, v)) / len(targets)


def parse_refs(js: str) -> List[Dict[str, Any]]:
    try:
        refs = json.loads(js)
        return refs if isinstance(refs, list) else []
    except Exception:
        return []


def target_rank(row, targets: List[str]) -> int:
    refs = parse_refs(row.get("retrieved_references_json", "[]"))
    for r in refs:
        blob = " ".join([
            str(r.get("reference", "")),
            str(r.get("pasal_id", "")),
            str(r.get("source_file", "")),
            str(r.get("text_preview", "")),
        ])
        if contains_any(blob, targets):
            return int(r.get("rank", 999))
    return 999


def reciprocal_rank(rank: int) -> float:
    return 0.0 if rank >= 999 else 1.0 / rank


def precision_at_k(row, targets: List[str], k: int = 3) -> float:
    refs = parse_refs(row.get("retrieved_references_json", "[]"))[:k]
    if not refs:
        return 0.0
    hits = 0
    for r in refs:
        blob = " ".join([str(r.get("reference", "")), str(r.get("pasal_id", "")), str(r.get("source_file", "")), str(r.get("text_preview", ""))])
        hits += contains_any(blob, targets)
    return hits / len(refs)


def answer_length_score(answer: str) -> float:
    words = len(str(answer).split())
    if words < 25:
        return words / 25
    if words <= 220:
        return 1.0
    return max(0.3, 1.0 - ((words - 220) / 300))

kw = df.apply(lambda r: keyword_hits(r.get("answer", ""), r.get("expected_keywords", "")), axis=1)
df["keyword_hits"] = kw.apply(lambda x: "; ".join(x["hits"]))
df["keyword_missing"] = kw.apply(lambda x: "; ".join(x["missing"]))
df["keyword_coverage"] = kw.apply(lambda x: x["score"])

df["expected_article_rank"] = df.apply(lambda r: target_rank(r, split_targets(r.get("expected_articles", ""))), axis=1)
df["expected_law_rank"] = df.apply(lambda r: target_rank(r, split_targets(r.get("expected_law_numbers", ""))), axis=1)
df["retrieval_citation_coverage"] = df.apply(lambda r: target_coverage(r.get("retrieved_references_text", ""), split_targets(r.get("expected_citations", ""))), axis=1)
df["retrieval_citation_hit"] = (df["retrieval_citation_coverage"] > 0).astype(int)
df["retrieval_article_hit"] = (df["expected_article_rank"] < 999).astype(int)
df["retrieval_law_hit"] = (df["expected_law_rank"] < 999).astype(int)
df["article_hit_at_3"] = (df["expected_article_rank"] <= 3).astype(int)
df["article_hit_at_8"] = (df["expected_article_rank"] <= 8).astype(int)
df["article_mrr"] = df["expected_article_rank"].apply(reciprocal_rank)
df["law_mrr"] = df["expected_law_rank"].apply(reciprocal_rank)
df["precision_at_3"] = df.apply(lambda r: precision_at_k(r, split_targets(r.get("expected_articles", "")) + split_targets(r.get("expected_law_numbers", "")), k=3), axis=1)
df["answer_citation_hit"] = df.apply(lambda r: contains_any(r.get("answer", ""), split_targets(str(r.get("expected_citations", "")) + ";" + str(r.get("expected_articles", "")) + ";" + str(r.get("expected_law_numbers", "")))), axis=1)
df["answer_word_count"] = df["answer"].fillna("").apply(lambda x: len(str(x).split()))
df["answer_length_score"] = df["answer"].apply(answer_length_score)

df[["id", "keyword_coverage", "retrieval_citation_coverage", "retrieval_law_hit", "retrieval_article_hit", "article_mrr", "precision_at_3", "answer_citation_hit"]].head()

In [ ]:
sem_model = SentenceTransformer(SEMANTIC_MODEL, device=DEVICE)

expected = df["expected_answer"].fillna("").tolist()
answers = df["answer"].fillna("").tolist()
emb_expected = sem_model.encode(expected, normalize_embeddings=True, show_progress_bar=True)
emb_answers = sem_model.encode(answers, normalize_embeddings=True, show_progress_bar=True)
df["semantic_similarity"] = [float(np.dot(a, b)) for a, b in zip(emb_answers, emb_expected)]
df[["id", "semantic_similarity"]].head()

## LLM-as-a-Judge (Evaluation)

Bagian ini menambahkan evaluasi menggunakan LLM sebagai hakim ("judge"). Karena menjalankan evaluasi LLM pada seluruh dataset bisa memakan waktu lama, kita menyediakan dua pendekatan:
1. **Custom LLM Judge**: Menggunakan model yang sudah dimuat (local) untuk memberikan skor kualitas secara kualitatif.
2. **Ragas**: Menyiapkan dataset dalam format RAGAS untuk evaluasi metrik *faithfulness*, *relevance*, dll.

In [ ]:
def llm_judge_grade(question: str, context: str, answer: str, expected_answer: str) -> float:
    judge_prompt = f"""Kamu adalah instruktur hukum ahli. Berikan nilai pada kualitas jawaban asisten berdasarkan referensi dan kunci jawaban.
NILAI dalam angka 0 sampai 100 (0: sangat buruk/salah, 100: sempurna).

KRITERIA:
1. Akurasi hukum dibandingkan kunci jawaban.
2. Kejelasan dan peringkasan informasi.
3. Apakah jawaban didukung oleh referensi yang diberikan?

REFERENSI:
{context}

KUNCI JAWABAN:
{expected_answer}

JAWABAN ASISTEN:
{answer}

Format output: Hanya angka (0-100). Contoh: 85
NILAI:"""

    messages = [{"role": "system", "content": "Berikan nilai numerik saja."}, {"role": "user", "content": judge_prompt}]
    try:
        raw_score = generate_chat_text(messages, max_new_tokens=10)
        # Ambil angka pertama yang ditemukan
        matches = re.findall(r"\d+", raw_score)
        if matches:
            score = float(matches[0])
            return min(max(score / 100.0, 0.0), 1.0)
    except Exception as e:
        print(f"Judge error: {e}")
    return 0.5 # Fallback score

print("Menjalankan LLM-as-a-Judge pada seluruh dataset...")
judge_scores = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="LLM Judge"):
    # Re-assemble context for judge if not stored
    refs = json.loads(row["retrieved_references_json"])
    docs = [{"text": r["text_preview"], "metadata": {"citation_text": r["reference"]}} for r in refs]
    context_text = "\n".join([f"- {d['metadata']['citation_text']}: {d['text']}" for d in docs])
    
    score = llm_judge_grade(row["question"], context_text, row["answer"], row["expected_answer"])
    judge_scores.append(score)

df["llm_judge_score"] = judge_scores
df[["id", "llm_judge_score", "semantic_similarity"]].head()

In [ ]:
from datasets import Dataset

def prepare_ragas_dataset(df_eval):
    data = {
        "question": df_eval["question"].tolist(),
        "answer": df_eval["answer"].tolist(),
        "contexts": [],
        "ground_truth": df_eval["expected_answer"].tolist()
    }
    
    for _, row in df_eval.iterrows():
        refs = json.loads(row["retrieved_references_json"])
        data["contexts"].append([r["text_preview"] for r in refs])
    
    return Dataset.from_dict(data)

ragas_ds = prepare_ragas_dataset(df)
print("Ragas dataset prepared with", len(ragas_ds), "samples.")
# Contoh penggunaan jika OpenAI Key tersedia:
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy
# result = evaluate(ragas_ds, metrics=[faithfulness, answer_relevancy])
# print(result)

In [ ]:
weights = {
    "llm_judge_score": 0.20,
    "semantic_similarity": 0.15,
    "keyword_coverage": 0.15,
    "retrieval_citation_coverage": 0.15,
    "retrieval_article_hit": 0.10,
    "retrieval_law_hit": 0.05,
    "answer_citation_hit": 0.10,
    "article_hit_at_3": 0.05,
    "article_mrr": 0.03,
    "precision_at_3": 0.01,
    "answer_length_score": 0.01,
}

df["overall_score"] = sum(df[k].clip(0, 1) * w for k, w in weights.items())
df["quality_label"] = pd.cut(
    df["overall_score"],
    bins=[-0.01, 0.50, 0.70, 0.85, 1.01],
    labels=["poor", "fair", "good", "excellent"]
)

metric_cols = list(weights.keys()) + ["article_hit_at_8", "law_mrr", "overall_score", "latency_seconds", "answer_word_count"]
summary = pd.DataFrame({
    "metric": metric_cols,
    "mean": [df[k].mean() for k in metric_cols],
    "median": [df[k].median() for k in metric_cols],
    "min": [df[k].min() for k in metric_cols],
    "max": [df[k].max() for k in metric_cols],
})

by_topic = df.groupby("topic", dropna=False).agg(
    question_count=("id", "count"),
    overall_score=("overall_score", "mean"),
    llm_judge_score=("llm_judge_score", "mean"),
    semantic_similarity=("semantic_similarity", "mean"),
    keyword_coverage=("keyword_coverage", "mean"),
    retrieval_citation_coverage=("retrieval_citation_coverage", "mean"),
    retrieval_article_hit=("retrieval_article_hit", "mean"),
    answer_citation_hit=("answer_citation_hit", "mean"),
    latency_seconds=("latency_seconds", "mean"),
).reset_index()

failure_review = df.sort_values("overall_score").loc[:, [
    "id", "topic", "question", "answer", "expected_answer", "overall_score", "llm_judge_score",
    "quality_label", "keyword_missing", "expected_article_rank", "expected_law_rank", "top1_reference"
]]
summary

In [ ]:
eval_csv = OUTPUT_DIR / "evaluation_table.csv"
summary_csv = OUTPUT_DIR / "evaluation_summary.csv"
by_topic_csv = OUTPUT_DIR / "evaluation_by_topic.csv"
failure_csv = OUTPUT_DIR / "failure_review.csv"
excel_path = OUTPUT_DIR / "evaluation_report.xlsx"

ordered_cols = [
    "id", "topic", "question", "expected_answer", "answer", "overall_score", "quality_label",
    "llm_judge_score", "semantic_similarity", "keyword_coverage", "keyword_hits", "keyword_missing",
    "retrieval_citation_coverage", "retrieval_citation_hit",
    "retrieval_law_hit", "retrieval_article_hit", "answer_citation_hit",
    "expected_article_rank", "expected_law_rank", "article_mrr", "law_mrr", "precision_at_3",
    "article_hit_at_3", "article_hit_at_8", "answer_word_count", "answer_length_score",
    "latency_seconds", "model_id", "top1_reference", "retrieved_references_text", "retrieved_references_json",
]
ordered_cols = [c for c in ordered_cols if c in df.columns]

df[ordered_cols].to_csv(eval_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
by_topic.to_csv(by_topic_csv, index=False, encoding="utf-8-sig")
failure_review.to_csv(failure_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df[ordered_cols].to_excel(writer, index=False, sheet_name="detail")
    summary.to_excel(writer, index=False, sheet_name="summary")
    by_topic.to_excel(writer, index=False, sheet_name="by_topic")
    failure_review.to_excel(writer, index=False, sheet_name="failure_review")

for p in [eval_csv, summary_csv, by_topic_csv, failure_csv, excel_path]:
    print("saved", p.resolve())

In [ ]:
def save_fig(name: str):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.resolve())
    plt.show()
    plt.close()

plot_metric_cols = [
    "llm_judge_score", "semantic_similarity", "keyword_coverage", "retrieval_citation_coverage", "retrieval_law_hit", "retrieval_article_hit",
    "answer_citation_hit", "article_hit_at_3", "article_mrr", "precision_at_3", "answer_length_score"
]

plt.figure(figsize=(12, 6))
plt.bar(df["id"].astype(str), df["overall_score"])
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.title("Overall Score per Question")
plt.xlabel("Question ID")
plt.ylabel("Score")
save_fig("01_overall_score_per_question.png")

plt.figure(figsize=(8, 5))
plt.hist(df["overall_score"], bins=np.linspace(0, 1, 11), edgecolor="black")
plt.xlim(0, 1)
plt.title("Overall Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")
save_fig("02_overall_score_distribution.png")

metric_means = df[plot_metric_cols].mean().sort_values()
plt.figure(figsize=(10, 6))
plt.barh(metric_means.index, metric_means.values)
plt.xlim(0, 1)
plt.title("Average Metric Scores")
plt.xlabel("Mean Score")
save_fig("03_average_metric_scores.png")

plt.figure(figsize=(10, 5))
topic_scores = by_topic.sort_values("overall_score")
plt.barh(topic_scores["topic"].astype(str), topic_scores["overall_score"])
plt.xlim(0, 1)
plt.title("Average Overall Score by Topic")
plt.xlabel("Mean Overall Score")
save_fig("04_score_by_topic.png")

plt.figure(figsize=(8, 5))
plt.scatter(df["latency_seconds"], df["overall_score"])
for _, r in df.iterrows():
    plt.text(r["latency_seconds"], r["overall_score"], str(r["id"]), fontsize=8)
plt.ylim(0, 1)
plt.title("Latency versus Overall Score")
plt.xlabel("Latency seconds")
plt.ylabel("Overall Score")
save_fig("05_latency_vs_score.png")

heat = df.set_index("id")[plot_metric_cols]
plt.figure(figsize=(12, 7))
plt.imshow(heat.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(plot_metric_cols)), plot_metric_cols, rotation=45, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="Score")
plt.title("Metric Heatmap per Question")
save_fig("06_metric_heatmap.png")

label_counts = df["quality_label"].value_counts().reindex(["poor", "fair", "good", "excellent"]).fillna(0)
plt.figure(figsize=(7, 5))
plt.bar(label_counts.index.astype(str), label_counts.values)
plt.title("Quality Label Count")
plt.xlabel("Quality")
plt.ylabel("Count")
save_fig("07_quality_label_count.png")

plt.figure(figsize=(12, 5))
rank_values = df["expected_article_rank"].replace(999, np.nan)
plt.bar(df["id"].astype(str), rank_values.fillna(9))
plt.axhline(3, linestyle="--")
plt.xticks(rotation=45, ha="right")
plt.title("Rank of Expected Article in Retrieved References")
plt.xlabel("Question ID")
plt.ylabel("Rank. Value 9 means not found")
save_fig("08_expected_article_rank.png")

plt.figure(figsize=(8, 5))
plt.scatter(df["semantic_similarity"], df["keyword_coverage"])
for _, r in df.iterrows():
    plt.text(r["semantic_similarity"], r["keyword_coverage"], str(r["id"]), fontsize=8)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.title("Semantic Similarity versus Keyword Coverage")
plt.xlabel("Semantic similarity")
plt.ylabel("Keyword coverage")
save_fig("09_semantic_vs_keyword.png")

plt.figure(figsize=(10, 5))
plt.boxplot([df.loc[df["topic"] == t, "overall_score"] for t in sorted(df["topic"].dropna().unique())], labels=sorted(df["topic"].dropna().unique()), vert=False)
plt.xlim(0, 1)
plt.title("Overall Score Spread by Topic")
plt.xlabel("Overall score")
save_fig("10_score_spread_by_topic.png")

plt.figure(figsize=(12, 5))
plt.bar(df["id"].astype(str), df["answer_word_count"])
plt.xticks(rotation=45, ha="right")
plt.title("Answer Word Count per Question")
plt.xlabel("Question ID")
plt.ylabel("Words")
save_fig("11_answer_word_count.png")

plt.figure(figsize=(12, 5))
bottom = np.zeros(len(df))
for col in ["retrieval_law_hit", "retrieval_article_hit", "answer_citation_hit", "article_hit_at_3"]:
    plt.bar(df["id"].astype(str), df[col], bottom=bottom, label=col)
    bottom += df[col].values
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 4)
plt.title("Binary Retrieval and Citation Hits")
plt.xlabel("Question ID")
plt.ylabel("Hit count")
plt.legend()
save_fig("12_binary_hit_stack.png")

In [ ]:
worst = failure_review.head(5)
report = f"""# Laporan Evaluasi RAG

Jumlah soal: {len(df)}
Model: {MODEL_ID}

## Ringkasan

- Rata-rata overall score: {df['overall_score'].mean():.3f}
- Median overall score: {df['overall_score'].median():.3f}
- Rata-rata LLM Judge score: {df['llm_judge_score'].mean():.3f}
- Rata-rata semantic similarity: {df['semantic_similarity'].mean():.3f}
- Keyword coverage: {df['keyword_coverage'].mean():.3f}
- Retrieval citation coverage: {df['retrieval_citation_coverage'].mean():.3f}
- Retrieval citation hit rate: {df['retrieval_citation_hit'].mean():.3f}
- Retrieval law hit rate: {df['retrieval_law_hit'].mean():.3f}
- Retrieval article hit rate: {df['retrieval_article_hit'].mean():.3f}
- Article hit@3: {df['article_hit_at_3'].mean():.3f}
- Article MRR: {df['article_mrr'].mean():.3f}
- Precision@3: {df['precision_at_3'].mean():.3f}
- Answer citation hit rate: {df['answer_citation_hit'].mean():.3f}
- Rata-rata latency detik: {df['latency_seconds'].mean():.2f}

## Lima Skor Terendah

{worst[['id', 'topic', 'overall_score', 'llm_judge_score', 'keyword_missing', 'expected_article_rank', 'top1_reference']].to_markdown(index=False)}

## File Penting

1. ../data/eval_outputs/rag_inference_results.csv
2. ../data/eval_outputs/evaluation_table.csv
3. ../data/eval_outputs/evaluation_summary.csv
4. ../data/eval_outputs/evaluation_by_topic.csv
5. ../data/eval_outputs/failure_review.csv
6. ../data/eval_outputs/evaluation_report.xlsx
7. ../data/eval_outputs/plots
"""
report_path = OUTPUT_DIR / "evaluation_report.md"
report_path.write_text(report, encoding="utf-8")
print(report)
print("saved", report_path.resolve())